# Preprocessing Exploration

Load a sample image and step through each preprocessing stage visually.
Use this notebook to tune Canny thresholds, contour filtering, and perspective correction on your own images.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from config import load_config
from src.preprocessing import NoteDetector, BoundaryDetector, PerspectiveCorrector
from src.preprocessing.image_utils import load_image, resize_max_dim, to_gray, adaptive_canny_thresholds
from src.grid import GridSplitter
from src.utils import draw_contour, draw_grid

config = load_config()
config["debug"] = False

def show(img, title="", cmap=None):
    plt.figure(figsize=(12, 6))
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or "gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# 1. Load an image
IMAGE_PATH = "../data/samples/note.jpg"   # ← change me

img = load_image(IMAGE_PATH)
resized, scale = resize_max_dim(img, config["preprocessing"]["resize_max_dim"])
show(resized, f"Input (resized, scale={scale:.3f})")

In [ ]:
# 2. Edge detection
gray = to_gray(resized)
blurred = cv2.GaussianBlur(gray, tuple(config["preprocessing"]["gaussian_kernel"]),
                            config["preprocessing"]["gaussian_sigma"])
lower, upper = adaptive_canny_thresholds(blurred)
print(f"Adaptive Canny thresholds: lower={lower}, upper={upper}")
edges = cv2.Canny(blurred, lower, upper)
show(edges, "Canny edges")

In [ ]:
# 3. Detect the note boundary
detector = BoundaryDetector(config)
corners = detector.detect(resized)
if corners is None:
    print("⚠️  No note detected. Try lowering min_area_ratio or tweaking Canny.")
else:
    vis = draw_contour(resized, corners.reshape(-1, 1, 2))
    show(vis, "Detected boundary")

In [ ]:
# 4. Perspective correction
corrector = PerspectiveCorrector(config)
aligned = corrector.correct_auto_orient(resized, corners)
show(aligned, "Aligned (top-down)")

In [ ]:
# 5. Grid overlay
grid_vis = draw_grid(aligned, rows=3, cols=3)
show(grid_vis, "3×3 grid (patches numbered 1–9)")

In [ ]:
# 6. OCR patches
splitter = GridSplitter(config)
patches = splitter.get_patches_by_index(aligned, config["grid"]["ocr_patches"])

fig, axes = plt.subplots(1, len(patches), figsize=(15, 5))
for ax, p in zip(axes, patches):
    ax.imshow(cv2.cvtColor(p.image, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Patch {p.index}")
    ax.axis("off")
plt.show()